# 05 — Demo: Open-Vocabulary 3D Object Search

This notebook demonstrates the full pipeline on a real-world scan:

1. Load a point cloud scan (Polycam / in-the-wild)
2. Extract features with **Concerto** (frozen encoder)
3. Translate features with the trained **MLP head**
4. Enter a **free-text query** → get a heatmap on the 3D scene

**Author:** Adrian (Data preparation & demo)


## 0. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone / pull the repo
import os

REPO_PATH = '/content/Deep_learning_project'

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git pull

%cd {REPO_PATH}
import sys
sys.path.insert(0, REPO_PATH)

In [ ]:
# Install dependencies
!pip install open3d open_clip_torch -q

In [ ]:
# Core imports
import numpy as np
import torch
import plotly.graph_objects as go
from pathlib import Path

# Project imports
from src.dataset import LABEL_TEXT
from src.visualize import (
    visualize_point_cloud,
    visualize_heatmap,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 1. Load the Polycam Scan

We load the preprocessed point cloud produced by `export_polycam.py`.

In [ ]:
# ── Config: set your paths here ───────────────────────────────────────────────
POLYCAM_DIR  = Path('/content/drive/MyDrive/DL_Project/data/polycam_processed/room1')
CHECKPOINT   = Path('/content/drive/MyDrive/DL_Project/checkpoints/mlp_best.pth')  # trained MLP
# ─────────────────────────────────────────────────────────────────────────────

coord  = np.load(POLYCAM_DIR / 'coord.npy')   # (N, 3) XYZ
color  = np.load(POLYCAM_DIR / 'color.npy')   # (N, 3) RGB [0,1]
normal = np.load(POLYCAM_DIR / 'normal.npy')  # (N, 3) normals

print(f'Loaded scan: {len(coord):,} points')
print(f'XYZ range : {coord.min():.3f} to {coord.max():.3f}')
print(f'RGB range : {color.min():.3f} to {color.max():.3f}')

In [ ]:
# Visualise the raw scan
fig = visualize_point_cloud(
    points=coord,
    colors=color,
    title='Polycam Scan — Raw Point Cloud',
    point_size=1.5,
)
fig.show()

## 2. Extract Concerto Features

We pass the point cloud through the **frozen** Concerto encoder to get per-point 3D features.

The encoder expects the processed room contract confirmed for the project: `coord`, `color`, and `normal`.

In [ ]:
# ── Check if encoder is available ─────────────────────────────────────────────
import importlib.util

encoder_available = importlib.util.find_spec('src.encoder') is not None

if encoder_available:
    from src.encoder import ConcertoEncoder

    print('Loading Concerto encoder...')
    encoder = ConcertoEncoder(device=DEVICE)
    encoder.eval()

    encoder_input = {
        'coord': coord,
        'color': color,
        'normal': normal,
    }

    print('Extracting features...')
    with torch.no_grad():
        features_3d = encoder(encoder_input)  # (N, 256)

    print(f'3D features shape: {features_3d.shape}')

else:
    print('⚠️  src/encoder.py not found yet.')
    print('    Generating RANDOM features as placeholder (shape: N x 256)')
    print('    Replace this with real Concerto features when encoder.py is pushed.')

    N = len(coord)
    features_3d = torch.randn(N, 256).to(DEVICE)  # placeholder
    print(f'Placeholder features shape: {features_3d.shape}')

## 3. Apply the MLP Translation Head

The trained MLP maps Concerto's 3D features into CLIP's text embedding space.

> ⚠️ **Placeholder** — waiting for Leonardo to push `src/translation_head.py`
> and a trained checkpoint to Drive.

In [ ]:
# ── Check if translation head is available ────────────────────────────────────
translation_available = (
    importlib.util.find_spec('src.translation_head') is not None
    and CHECKPOINT.exists()
)

if translation_available:
    from src.translation_head import MLPTranslationHead

    print('Loading trained MLP head...')
    mlp = MLPTranslationHead(input_dim=256, output_dim=512)
    mlp.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
    mlp = mlp.to(DEVICE)
    mlp.eval()

    print('Translating 3D features into CLIP space...')
    with torch.no_grad():
        features_clip = mlp(features_3d)  # (N, 512)

    # L2-normalise for cosine similarity
    features_clip = torch.nn.functional.normalize(features_clip, dim=-1)
    print(f'Translated features shape: {features_clip.shape}')

else:
    print('⚠️  src/translation_head.py or checkpoint not found yet.')
    print('    Using RANDOM CLIP-space features as placeholder (shape: N x 512)')

    N = len(coord)
    features_clip = torch.nn.functional.normalize(
        torch.randn(N, 512).to(DEVICE), dim=-1
    )
    print(f'Placeholder features shape: {features_clip.shape}')

## 4. Text Query → Heatmap

Enter any free-text query. CLIP embeds it and we compute cosine similarity
against all per-point features to produce a heatmap.

In [ ]:
import open_clip

# Load CLIP text encoder (frozen)
print('Loading CLIP text encoder...')
clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_model = clip_model.to(DEVICE)
clip_model.eval()
tokenizer = open_clip.get_tokenizer('ViT-B-32')
print('CLIP ready.')

In [ ]:
def query_scene(text_query: str, top_percent: float = 10.0):
    """
    Given a text query, compute cosine similarity against all per-point
    CLIP-space features and visualise a heatmap on the point cloud.

    Args:
        text_query   : free-text query, e.g. 'chair', 'window', 'table'
        top_percent  : highlight the top X% most similar points (default: 10%)
    """
    print(f'\nQuery: "{text_query}"')

    # 1. Embed the text query with CLIP
    tokens = tokenizer([text_query]).to(DEVICE)
    with torch.no_grad():
        text_emb = clip_model.encode_text(tokens)           # (1, 512)
        text_emb = torch.nn.functional.normalize(text_emb, dim=-1)

    # 2. Cosine similarity: (N, 512) · (512, 1) → (N,)
    similarity = (features_clip @ text_emb.T).squeeze(-1)  # (N,)
    similarity = similarity.cpu().numpy()

    # 3. Normalise similarity to [0, 1] for visualisation
    sim_min, sim_max = similarity.min(), similarity.max()
    sim_norm = (similarity - sim_min) / (sim_max - sim_min + 1e-8)

    print(f'Similarity range: {sim_min:.3f} to {sim_max:.3f}')
    print(f'Highlighting top {top_percent}% of points')

    # 4. Visualise heatmap
    threshold = np.percentile(sim_norm, 100 - top_percent)
    highlight_mask = sim_norm >= threshold
    print(f'Highlighted points: {highlight_mask.sum():,} / {len(coord):,}')

    # Build color array: highlighted = red, rest = gray
    heatmap_colors = np.full((len(coord), 3), 0.6)         # gray background
    heatmap_colors[highlight_mask] = [1.0, 0.2, 0.2]       # red highlights

    fig = go.Figure(data=[go.Scatter3d(
        x=coord[:, 0],
        y=coord[:, 1],
        z=coord[:, 2],
        mode='markers',
        marker=dict(
            size=1.5,
            color=[f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'
                   for r, g, b in heatmap_colors],
            opacity=0.8,
        )
    )])

    fig.update_layout(
        title=f'Query: "{text_query}" — top {top_percent}% matches highlighted in red',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            bgcolor='rgb(20,20,20)',
        ),
        paper_bgcolor='rgb(30,30,30)',
        font_color='white',
        height=700,
    )
    fig.show()
    return sim_norm

In [ ]:
# ── Try different queries here ─────────────────────────────────────────────────
sim = query_scene('chair', top_percent=10)

In [ ]:
sim = query_scene('floor', top_percent=20)

In [ ]:
sim = query_scene('wall', top_percent=15)

In [ ]:
# ── Interactive: type your own query ──────────────────────────────────────────
your_query = input('Enter your query (e.g. "table", "window", "door"): ')
sim = query_scene(your_query, top_percent=10)

## 5. Compare Multiple Queries Side by Side

In [ ]:
def compare_queries(queries: list[str], top_percent: float = 10.0):
    """
    Run multiple queries and show which points each query highlights.
    Each query gets a distinct color.
    """
    query_colors = [
        [1.0, 0.2, 0.2],   # red
        [0.2, 0.8, 0.2],   # green
        [0.2, 0.4, 1.0],   # blue
        [1.0, 0.8, 0.0],   # yellow
    ]

    final_colors = np.full((len(coord), 3), 0.4)  # dark gray background

    for i, query in enumerate(queries):
        tokens = tokenizer([query]).to(DEVICE)
        with torch.no_grad():
            text_emb = clip_model.encode_text(tokens)
            text_emb = torch.nn.functional.normalize(text_emb, dim=-1)

        sim = (features_clip @ text_emb.T).squeeze(-1).cpu().numpy()
        sim_norm = (sim - sim.min()) / (sim.max() - sim.min() + 1e-8)
        threshold = np.percentile(sim_norm, 100 - top_percent)
        mask = sim_norm >= threshold
        final_colors[mask] = query_colors[i % len(query_colors)]
        print(f'  "{query}": {mask.sum():,} points highlighted')

    fig = go.Figure(data=[go.Scatter3d(
        x=coord[:, 0], y=coord[:, 1], z=coord[:, 2],
        mode='markers',
        marker=dict(
            size=1.5,
            color=[f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'
                   for r, g, b in final_colors],
            opacity=0.8,
        )
    )])

    legend_str = '  |  '.join(
        [f'"{q}"' for q in queries]
    )
    fig.update_layout(
        title=f'Multi-query comparison: {legend_str}',
        scene=dict(bgcolor='rgb(20,20,20)'),
        paper_bgcolor='rgb(30,30,30)',
        font_color='white',
        height=700,
    )
    fig.show()


# Compare multiple queries at once
compare_queries(['chair', 'floor', 'wall'], top_percent=10)

## 6. Notes for the Presentation

- **Section 2** (Concerto features) will use real features once `src/encoder.py` is pushed by Ricardo/Leonardo
- **Section 3** (MLP head) will use the trained checkpoint once training is complete
- Currently running with **placeholder random features** — heatmaps will be random until real features are plugged in
- The `query_scene()` and `compare_queries()` functions require **no changes** once real features are available
- To demo live: run all cells up to Section 4, then use the interactive input cell
